# Training a Small Language Model

Actually finetuning.

We will finetune [GPT2](https://huggingface.co/openai-community/gpt2), the first "groundbreaking" transformer language model (from 2019, no less). It is a really small Language Model (0.1 billion parameters).

And we will finetune it on an [english dictionary definitions dataset](https://huggingface.co/datasets/MAKILINGDING/english_dictionary).

## Outline

1.   Install libraries and setup the dataset
2.   Load the model (and test it)
3.   Train the model on the dataset (and test it)



# Setup

Install the required libraries

In [ ]:
!pip install -q transformers datasets

Then we setup the libraries to avoid showing "unnecessary" warnings

In [ ]:
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()   # only show errors

from datasets import logging as ds_logging
ds_logging.set_verbosity_error()   # only show errors


Select the training dataset. In this example we are going to use an [english dictionary dataset](https://huggingface.co/datasets/MAKILINGDING/english_dictionary)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("MAKILINGDING/english_dictionary")

print(dataset)

Then we split the dataset in "train" and "test"

In [ ]:
dataset = dataset["train"].train_test_split(test_size=0.1, shuffle=True)

print(dataset)

Then the dataset is processed and converted to a format that the neural network understands (from words to tokens). This process is dependant on the chosen neural network.

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize(examples):
    return tokenizer(examples["definition"], truncation=True, padding='max_length', max_length=512)

# .map() processes all the rows in the dataset
dataset = dataset.map(tokenize, batched=True)

# Loading the model

Then we need to load the model, and for training we can't use a pipeline, so we load the model directly.

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").to("cuda")

Then we define a function to generate text that we will reuse later (you don't need to understand it).

In [ ]:
def generate_text(prompt, model, tokenizer, max_length=512, temperature=1, top_k=50, top_p=0.95):
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        inputs,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True
    )

    generated_text = tokenizer.decode(outputs[0].cpu(), skip_special_tokens=True)
    return generated_text

This a simple text generation neural network, so we can't give it instructions, but just the starting text.

In [ ]:
generated_text = generate_text("Unce upon a time", model, tokenizer)

print(generated_text)

# Training the model

First we define some parameters you don't need to understand, except 'num_train_epochs'. 'num_train_epochs' defines how many times does the model look at the dataset before finishing training. In this case we set it up at 0.1 (so just 10%) because we don't have time, but it should be at least over 1 (and )

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
  output_dir='./results',
  num_train_epochs=0.1,
  per_device_train_batch_size=8,
  per_device_eval_batch_size=8,
  eval_strategy="steps",
  load_best_model_at_end=True,
  metric_for_best_model="eval_loss",
  save_steps=500,
  logging_steps=100,
  eval_steps=100,
  dataloader_pin_memory=False
)

And then we start training the model and printing the loss. With the parameters we have defined it should take around 30 minutes.

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=dataset['train'],
  eval_dataset=dataset['test'],
  data_collator=data_collator
)

trainer.train(resume_from_checkpoint=False)

Once it has been trained we can generate new text.

In [ ]:
generated_text = generate_text("A group of students", model, tokenizer)

print(generated_text)

# Finalizing

When you finish working you have to remember to **stop the runtime**, because there is a time limit and to avoid wasting resources. To stop the runtime click Manage Sessions on the Runtime menu. Once the dialog opens click terminate on the current runtime.

> But when you stop the runtime everything you have not saved is ⚠ **lost** ⚠, so be sure to **download** everything you want to keep before stopping it.


# Credits

Taller Estampa https://tallerestampa.com / https://github.com/estampa

### Based on

[Sentence Transformers](https://github.com/UKPLab/sentence-transformers/tree/master/examples/applications/image-search)
